# Object arrival/departure detection

**Recipe 86** · Level 2 (Easy) · Decision type: `Choice`

Determine whether a specified package, vehicle, or other object appeared, disappeared, remained, or is uncertain across ordered still frames of a scene.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build a small observer for a blue-taped parcel on a delivery mat. A `Choice` classifies the entire ordered sequence as `arrival`, `departure`, `unchanged`, or `uncertain`; Python checks the stills and controls whether a visible-event note can be simulated. Images are synthetic Pillow drawings, not photographs. All answers are authored synthetic fixtures, so the numbers below check the pipeline and are not Jev results.

Here **arrival** means absent then visible in the sampled images, and **departure** means visible then absent. Neither proves a delivery, removal, or theft. An occluder can hide a parcel without moving it. An object can leave and return between sampled frames; even an `unchanged` answer makes no claim about unobserved time. The images feature consumes PNG/JPEG stills, not native video ([S09](https://docs.typesafe.ai/images)).

## Setup and run mode

A [replay key](../../docs/glossary.md#replay-key) identifies a request by its state, questions, and image bytes. The default backend replays stored answers without a key or network. The same definitions can use experimental image inference with pinned `jev-1.14.0`; see the README for a separate recording workflow. Live exploration alone produces no reportable held-out result.

There are 12 validation clips, 12 test clips, and two unscored demonstrations. Each clip supplies three timestamped stills. Each eligible clip consumes at most one request; repeated inspection reuses its answer.

In [ ]:
from pathlib import Path

from IPython.display import display

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import (
    accuracy,
    confusion_matrix,
    evaluate_outcomes,
    select_confidence_threshold,
)
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

ROOT = Path.cwd()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
runner = helpers.ClipRunner(backend, max_requests=20)
scored = [e for e in examples if e.split in {"validation", "test"}]
check = " -- a pipeline check, not a Jev result" if backend.mode == "synthetic" else ""
selection = " -- validation selection step"
apply_style()
run_header(86, "Object arrival/departure detection", backend=backend, n_examples=len(scored))

## The state

The request contains only the target description, chronological timestamps in seconds, and opaque image references. Relative filenames stay in local input fixtures. Gold labels, split names, and drawing specifications are never sent. Python rejects negative, duplicate, reversed, or nonfinite timestamps rather than silently sorting them.

Sampling is fixed before selection: three or four frames, gaps no larger than two seconds, and a span no longer than four seconds. Camera checks compare the upper wall strip to the first frame; pixel brightness and contrast reject dark or flat images. These simple checks assume a fixed camera and an empty wall strip. They do not certify visibility or detect every bad frame. These are already sampled stills; a real video decoder must preserve actual capture times.

In [ ]:
demo = next(e for e in examples if e.id == "d01")
prepared = helpers.prepare(demo.fields, ROOT)
print(prepared.state)
print("Local quality findings:", prepared.issues)
display(helpers.contact_sheet(demo.fields, ROOT))

## The questions

One bounded question asks about one target's visible history over the whole sequence. Asking each endpoint separately would miss a disappearance and return in the middle. Independent targets could be batched as separate questions in one request; this recipe has only one target. The fixed `uncertain` option covers occlusion, ambiguous identity and multiple transitions.

In [ ]:
questions = helpers.build_questions()
print(questions["event"].instructions)
for option, meaning in questions["event"].criteria.items():
    print(f"{option}: {meaning}")

## One answer up close

The typed answer includes the choice, its distribution, and a distribution-derived confidence ([S03](https://docs.typesafe.ai/confidence)). Confidence is not a guarantee of a correct event. [Provenance](../../docs/glossary.md#provenance) records where an answer came from; these answers are synthetic, not recorded model behavior.

In [ ]:
demo_observation = runner.observe(demo.fields, ROOT)
show_answer(demo_observation.answer)
print(run_record(demo_observation.result, mode=backend.mode))

### Why the middle frame matters

The next unscored example starts and ends with the parcel visible, but the middle frame is empty. The four-option task cannot represent both a departure and a return, so the correct response is `uncertain`. Python does not compress the clip into its endpoints.

In [ ]:
returned = next(e for e in examples if e.id == "d02")
display(helpers.contact_sheet(returned.fields, ROOT))
returned_observation = runner.observe(returned.fields, ROOT)
show_answer(returned_observation.answer)

## Python's part

A [gold label](../../docs/glossary.md#gold-label) is the authored expected judgment for a clip. A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a resolved answer only at or above a threshold. [Review](../../docs/glossary.md#review) means the question remains pending for a person; `uncertain` always counts as review, even with high confidence.

Choose the confidence threshold from validation alone. The candidate family is the distinct observed confidences of eligible, resolved validation answers. The objective is the lowest threshold attaining validation [accuracy](../../docs/glossary.md#accuracy), the fraction of correct accepted answers, of 1.0. This selection excludes unconditional quality blocks and `uncertain` answers; they still count in full-denominator reporting. If no candidate reaches the objective, selection raises and stops. Freeze the result before inspecting test predictions.

The runner rejects unusable clips before a request, caches repeated requests, and stops at 20 requests. The live backend also counts transport attempts and retries against its own budget. Accepted results can only append a simulated visible-event note with `physical_event='not established'`; unresolved results enter `ReviewQueue`. Nothing changes a delivery system.

In [ ]:
validation = [e for e in examples if e.split == "validation"]
validation_observations = {e.id: runner.observe(e.fields, ROOT) for e in validation}
eligible = [
    (e, validation_observations[e.id].answer)
    for e in validation
    if validation_observations[e.id].answer is not None
    and validation_observations[e.id].answer.choice != "uncertain"
]
threshold = select_confidence_threshold(
    [a.choice == labels[e.id] for e, a in eligible],
    [a.confidence for _, a in eligible],
    target_accuracy=1.0,
)
print(
    f"Frozen threshold={threshold:.4f}, eligible N={len(eligible)}, validation N={len(validation)}{selection}{check}"
)
validation_outcomes = [helpers.route(validation_observations[e.id], threshold) for e in validation]
validation_metrics = evaluate_outcomes(
    [o.accepted for o in validation_outcomes],
    [o.label == labels[e.id] for e, o in zip(validation, validation_outcomes, strict=True)],
)
print(
    f"Validation N={validation_metrics.n_total}, answered={validation_metrics.n_answered}, "
    f"accuracy={validation_metrics.accuracy:.4f}, coverage={validation_metrics.coverage:.4f}{selection}{check}"
)

## Evaluation

The frozen policy is now applied once to the 12 test clips. [Coverage](../../docs/glossary.md#coverage) is the fraction accepted; [risk](../../docs/glossary.md#risk) is the error fraction among those accepted. The shared `evaluate_outcomes` receives the actual accept/review flags, including quality checks and uncertainty, rather than a confidence-only approximation. Review is not a correct event decision.

A deterministic pixel baseline counts tan pixels in a fixed region of interest (the lower central mat), then examines all adjacent occupancy changes. Its constants are fixed in advance: at least 1,000 qualifying pixels indicates occupancy; one transition gives arrival/departure, two transitions give uncertainty. It shares the same input images and preflight checks. It does not read gold labels, row IDs, split names or drawing metadata. It cannot distinguish a brown crate from the specified parcel or reliably reason about occlusion.

Raw event accuracy includes the four-option `uncertain` category and preflight-blocked clips as uncertainty. Accepted event accuracy excludes all reviews; coverage shows that smaller denominator. The confusion matrix describes raw four-option judgments, not simulated actions.

In [ ]:
test = [e for e in examples if e.split == "test"]
test_observations = {e.id: runner.observe(e.fields, ROOT) for e in test}
actions, reviews = ActionLog(), ReviewQueue()
test_outcomes = [
    helpers.log_outcome(e.id, test_observations[e.id], threshold, actions, reviews) for e in test
]
gold = [labels[e.id] for e in test]
predicted = [
    test_observations[e.id].answer.choice
    if test_observations[e.id].answer is not None
    else "uncertain"
    for e in test
]
baseline = [helpers.pixel_baseline(test_observations[e.id].prepared) for e in test]
accepted = [o.accepted for o in test_outcomes]
metrics = evaluate_outcomes(accepted, [p == g for p, g in zip(predicted, gold, strict=True)])
baseline_metrics = evaluate_outcomes(
    [p != "uncertain" for p in baseline],
    [p == g for p, g in zip(baseline, gold, strict=True)],
)
if backend.mode == "live":
    print(
        "Unrecorded exploratory test run; no reportable held-out metrics. Record and replay first."
    )
else:
    print(f"Test N={len(test)}, raw event accuracy={accuracy(gold, predicted):.4f}{check}")
    print(
        f"Test N={len(test)}, accepted={metrics.n_answered}, review={len(reviews)}, "
        f"coverage={metrics.coverage:.4f}, abstention={1 - metrics.coverage:.4f}, "
        f"accepted event accuracy={metrics.accuracy:.4f}, risk={metrics.risk:.4f}{check}"
    )
    print(
        f"Pixel baseline N={len(test)}, raw event accuracy={accuracy(gold, baseline):.4f}, "
        f"accepted={baseline_metrics.n_answered}, coverage={baseline_metrics.coverage:.4f}, "
        f"accepted event accuracy={baseline_metrics.accuracy:.4f}, risk={baseline_metrics.risk:.4f}{check}"
    )
print("Requests used:", runner.requests, "of", runner.max_requests)
print("Simulated action sample:", actions.to_dicts()[:1])
print("Review queue:", [{"clip": r.item["clip"], "reason": r.reason} for r in reviews.pending()])

In [ ]:
if backend.mode != "live":
    matrix = confusion_matrix(gold, predicted, labels=list(helpers.OPTIONS))
    print(f"Test raw event matrix, N={len(test)}; rows=gold, columns=predicted{check}")
    print(matrix.labels)
    print(matrix.matrix)
    display(plot_confusion_matrix(matrix, title=f"Test event judgments, N={len(test)}{check}"))

### A mistake the frozen gate accepts

A brown crate disappears, while the specified blue-taped parcel is absent throughout. The synthetic answer incorrectly names `departure` with high confidence. It passes the frozen gate and creates a wrong simulated note. This example makes the limit observable: confidence and coarse camera checks cannot guarantee object identity. The baseline makes the same error for a different reason: its brown-pixel rule deliberately lacks identity recognition.

In [ ]:
wrong = [
    e for e, o in zip(test, test_outcomes, strict=True) if o.accepted and o.label != labels[e.id]
]
print("Accepted wrong test clips:", [e.id for e in wrong])
for example in wrong:
    display(helpers.contact_sheet(example.fields, ROOT))
    show_answer(test_observations[example.id].answer)
    print("Gold visible-history label:", labels[example.id])

## What was and was not measured

The cell reports the run mode and N in every mode, plus returned model and capture date for recorded results. All committed answers and images are synthetic. These metrics check composition, routing and evaluation on 24 authored clips; they say nothing about Jev quality on real camera footage. The two demos are unscored. There are no real video feeds, notifications, inventory changes or delivery decisions.

Latency, tokens and cost remain `None` when unknown. Replaying answers is not an inference latency measurement. The recorder preserves actual returned model, date and usage; absent latency measurements or billing stay unknown. The same two-second sampling rule can miss arbitrarily short events between stills, and the fixed camera strip can miss motion confined to another region. Real deployments need independently labeled footage, camera calibration and new held-out evaluation.

In [ ]:
print("Mode:", backend.mode)
if backend.mode == "recorded":
    print("Returned model:", backend.model, "capture dates:", sorted(backend.recorded_dates))
    print(
        "Recorded sample only; validation selected the gate and test was evaluated after freezing."
    )
    print(f"N: {len(scored)} scored (validation=12, test=12); demos=2 unscored")
elif backend.mode == "live":
    print(
        "Returned model:",
        backend.model,
        "capture dates:",
        sorted({r.date for r in backend.records}),
    )
    print("Unrecorded exploration; no reportable held-out model result.")
    print(f"N: {len(scored)} scored (validation=12, test=12); demos=2 unscored")
else:
    print("Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(scored)} scored (validation=12, test=12); demos=2 unscored{check}")
print("Example accounting:", run_record(demo_observation.result, mode=backend.mode))

## Next steps

Try a second target and batch the independent target judgments. For longer clips, define a bounded event vocabulary or segment the sequence in Python, then freeze a new policy on a separate validation split. Record real examples before making model-quality claims.

See [Sentiment classification](../01-sentiment-classification/) for the typed `Choice` and frozen-gate pattern, [Tool outcome verification](../29-tool-outcome-verification/) for evidence-limited claims, and [Abstention and review queues](../38-abstention-and-review-queues/) for explicit review accounting.